# 16 · Guardrails: immutability & anti-reward-hacking

Give a capable agent a score and a shell and the shortest path is often to edit
the scorer. Every guardrail exists for that. We exercise the load-bearing one:
the immutability hash guard.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


In [2]:
from harness.loop import snapshot_immutable, verify_immutable
from harness import IMMUTABLE_FILES, MODIFIABLE_FILES
print("immutable (agent CANNOT touch):"); [print("  ", f) for f in IMMUTABLE_FILES]
print("modifiable (agent CAN edit):");    [print("  ", f) for f in MODIFIABLE_FILES]

snap = snapshot_immutable()
verify_immutable(snap)                      # clean iteration -> passes
print("\nclean iteration: guard passes")

immutable (agent CANNOT touch):
   program.md
   harness/evaluator.py
   harness/data.py
   harness/loop.py
   harness/config.py
   harness/registry.py
modifiable (agent CAN edit):
   harness/train.py
   harness/model.py
   harness/mining.py

clean iteration: guard passes


In [3]:
# simulate the agent editing the evaluator mid-iteration
tampered = dict(snap); tampered["harness/evaluator.py"] = "0" * 64
try:
    verify_immutable(tampered)
except RuntimeError as e:
    print("REJECTED:", e)
print("\nThe score is thrown out -> you cannot win by editing the ruler.")

REJECTED: immutable files changed during the iteration; score rejected: harness/evaluator.py

The score is thrown out -> you cannot win by editing the ruler.


The other enforced rules: only the evaluator reads `test` (no peeking), a fixed
**budget** (no winning by training longer), and fixed seeds + a fixed test split
(a 'win' that's just variance reproduces away). The worst-group objective also
self-punishes *slice starvation* — over-fixing fog until another slice collapses
drops the score, so the loop reverts it.